# Adult Income

## Câu 5. Mã hóa one-hot

### Yêu cầu

Chuyển các cột phi số sang one-hot vector; đối chiếu trước và sau.

### Dữ liệu và phương pháp

Thuộc tính quyết định: income. Sử dụng toàn bộ dữ liệu được cung cấp. Giữ nguyên nguồn train/test sau khi loại dòng trống; nối hai tập để thực hiện quy trình của đề. Tương quan cao được quy ước là |r| ≥ 0,8 giữa các thuộc tính đầu vào; giữ cột đứng trước trong thứ tự gốc.

In [1]:
print("Đặng Ngọc Hiếu - 6451071024")

from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "datasets").is_dir():
    if (ROOT / "6451071024_Lab2" / "datasets").is_dir():
        ROOT = ROOT / "6451071024_Lab2"
        break
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Không tìm thấy thư mục datasets của LAB2.")
    ROOT = ROOT.parent
pd.set_option("display.max_columns", None)


train_raw = pd.read_csv(ROOT / "datasets/adult/adult.data.csv", skipinitialspace=True)
test_raw = pd.read_csv(ROOT / "datasets/adult/adult.test.csv", skipinitialspace=True)
if "income" not in train_raw or "income" not in test_raw:
    raise ValueError("Hai tệp Adult phải có cột income.")
if "fnlwgt" not in train_raw or "fnlwgt" in test_raw:
    raise ValueError("Kiểm tra hai phiên bản Adult: fnlwgt chỉ có trong tập huấn luyện.")


Đặng Ngọc Hiếu - 6451071024


In [2]:
print("Đặng Ngọc Hiếu - 6451071024")

# Kiểu string tường minh giúp nhận diện thiếu mà không dựa vào downcasting.
def clean_adult(frame):
    frame = frame.copy()
    for column in frame.select_dtypes(include=["object", "string"]).columns:
        values = frame[column].astype("string").str.strip()
        frame[column] = values.mask(values.isin(["?", ""]))
    return frame.dropna()

train = clean_adult(train_raw).drop(columns="fnlwgt")
test = clean_adult(test_raw)
if list(train.columns) != list(test.columns):
    raise ValueError("Tên và thứ tự cột Adult chưa khớp sau khi bỏ fnlwgt.")
n_train = len(train)
data = pd.concat([train, test], ignore_index=True)
TARGET = "income"


Đặng Ngọc Hiếu - 6451071024


In [3]:
print("Đặng Ngọc Hiếu - 6451071024")

# Tái hiện bước chọn thuộc tính của câu 3, không vẽ lại heatmap.
corr = data.drop(columns=TARGET).select_dtypes(include="number").corr(method="pearson")
CORRELATION_THRESHOLD = 0.8
# |r| >= 0,8 là quy ước đã nêu trong phương pháp, không phải ngưỡng đề ấn định.
drop_columns = []
for j, column in enumerate(corr.columns):
    for previous in corr.columns[:j]:
        value = corr.loc[previous, column]
        if previous not in drop_columns and pd.notna(value) and abs(value) >= CORRELATION_THRESHOLD:
            drop_columns.append(column)
            break
data = data.drop(columns=drop_columns)


Đặng Ngọc Hiếu - 6451071024


In [4]:
print("Đặng Ngọc Hiếu - 6451071024")

features = data.drop(columns=TARGET)
labels = data[TARGET]


Đặng Ngọc Hiếu - 6451071024


In [5]:
print("Đặng Ngọc Hiếu - 6451071024")

non_numeric = features.select_dtypes(exclude=["number", "bool"]).columns.tolist()
print("Các cột cần one-hot:", non_numeric)
print("Kiểu dữ liệu trước mã hóa:")
display(features.dtypes.to_frame("Kiểu dữ liệu"))
features_onehot = pd.get_dummies(features, columns=non_numeric, dtype="uint8")
print("Kích thước trước / sau:", features.shape, features_onehot.shape)
display(features.head(), features_onehot.head())
if features_onehot.select_dtypes(exclude=["number", "bool"]).shape[1]:
    raise ValueError("Đặc trưng đầu vào vẫn còn cột phi số.")
if not non_numeric:
    print("Các đặc trưng đã là số; không phát sinh cột one-hot.")


Đặng Ngọc Hiếu - 6451071024
Các cột cần one-hot: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']
Kiểu dữ liệu trước mã hóa:


,Kiểu dữ liệu
age,int64
workclass,string[python]
education,string[python]
education-num,int64
marital-status,string[python]
occupation,string[python]
relationship,string[python]
race,string[python]
sex,string[python]
capital-gain,int64


Kích thước trước / sau: (45222, 13) (45222, 103)


,age,workclass,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,39,State-gov,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States
1,50,Self-emp-not-inc,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States
2,38,Private,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States
3,53,Private,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States
4,28,Private,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba


,age,education-num,capital-gain,capital-loss,hours-per-week,workclass_Federal-gov,workclass_Local-gov,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,workclass_State-gov,workclass_Without-pay,education_10th,education_11th,education_12th,education_1st-4th,education_5th-6th,education_7th-8th,education_9th,education_Assoc-acdm,education_Assoc-voc,education_Bachelors,education_Doctorate,education_HS-grad,education_Masters,education_Preschool,education_Prof-school,education_Some-college,marital-status_Divorced,marital-status_Married-AF-spouse,marital-status_Married-civ-spouse,marital-status_Married-spouse-absent,marital-status_Never-married,marital-status_Separated,marital-status_Widowed,occupation_Adm-clerical,occupation_Armed-Forces,occupation_Craft-repair,occupation_Exec-managerial,occupation_Farming-fishing,occupation_Handlers-cleaners,occupation_Machine-op-inspct,occupation_Other-service,occupation_Priv-house-serv,occupation_Prof-specialty,occupation_Protective-serv,occupation_Sales,occupation_Tech-support,occupation_Transport-moving,relationship_Husband,relationship_Not-in-family,relationship_Other-relative,relationship_Own-child,relationship_Unmarried,relationship_Wife,race_Amer-Indian-Eskimo,race_Asian-Pac-Islander,race_Black,race_Other,race_White,sex_Female,sex_Male,native-country_Cambodia,native-country_Canada,native-country_China,native-country_Columbia,native-country_Cuba,native-country_Dominican-Republic,native-country_Ecuador,native-country_El-Salvador,native-country_England,native-country_France,native-country_Germany,native-country_Greece,native-country_Guatemala,native-country_Haiti,native-country_Holand-Netherlands,native-country_Honduras,native-country_Hong,native-country_Hungary,native-country_India,native-country_Iran,native-country_Ireland,native-country_Italy,native-country_Jamaica,native-country_Japan,native-country_Laos,native-country_Mexico,native-country_Nicaragua,native-country_Outlying-US(Guam-USVI-etc),native-country_Peru,native-country_Philippines,native-country_Poland,native-country_Portugal,native-country_Puerto-Rico,native-country_Scotland,native-country_South,native-country_Taiwan,native-country_Thailand,native-country_Trinadad&Tobago,native-country_United-States,native-country_Vietnam,native-country_Yugoslavia
0,39,13,2174,0,40,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
1,50,13,0,0,13,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
2,38,9,0,0,40,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
3,53,7,0,0,40,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
4,28,13,0,0,40,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
